<a href="https://colab.research.google.com/github/THE-KIRA/onnx/blob/main/reconfigurado_kira_Train_Your_First_Wake_Word_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Train Your First Custom Wake Word with Nanowakeword!

Welcome to the official tutorial for **Nanowakeword**!

In this notebook, we will guide you through the entire process of training a high-performance, custom wake word model from scratch. You don't need any pre-existing data—we will download everything we need and let Nanowakeword's intelligent engine do the heavy lifting.

**Our goal:** Go from zero to a ready-to-use wake word model in just a few simple steps. Let's get started!

**Installation**

In [7]:
# @title Step 1: Install Nanowakeword
# We install the full [train] package to get all the necessary dependencies.

# ! pip install --no-cache-dir "nanowakeword[train]==1.3.4" # will comeing
! pip install "nanowakeword[train] @ git+https://github.com/arcosoph/nanowakeword.git"
! pip install piper-tts

print("Installation complete!")

  Cloning https://github.com/arcosoph/nanowakeword.git to /tmp/pip-install-7t2gxtia/nanowakeword_ff79529fc16b47029fea34fd60911e38
  Running command git clone --filter=blob:none --quiet https://github.com/arcosoph/nanowakeword.git /tmp/pip-install-7t2gxtia/nanowakeword_ff79529fc16b47029fea34fd60911e38
  Resolved https://github.com/arcosoph/nanowakeword.git to commit 0cecbe11993198b17822930bcd9bdefbcc0932a9
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Installation complete!


## Step 2: Prepare the Dataset

A great model starts with great data. For this tutorial, we will:
1.  **Download** open-source noise and Room Impulse Response (RIR) datasets.
2.  **Organize** all your project files within a clean, well-structured folder hierarchy for better clarity and maintainability.

In [8]:
import shutil
import os, requests
from tqdm import tqdm
from datasets import load_dataset, Audio
from huggingface_hub import hf_hub_download
from concurrent.futures import ThreadPoolExecutor

# RIR data
# Disable HF symlink warning
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
# Output directory
output_dir = "data/rir"
os.makedirs(output_dir, exist_ok=True)

# Load dataset (streaming, no decode)
dataset = load_dataset(
    "davidscripka/MIT_environmental_impulse_responses",
    split="train",
    streaming=True
)
dataset = dataset.cast_column("audio", Audio(decode=False))

repo_id = "davidscripka/MIT_environmental_impulse_responses"

for row in tqdm(dataset):
    hf_path = row["audio"]["path"]
    filename = os.path.basename(hf_path)
    save_path = os.path.join(output_dir, filename)

    if os.path.exists(save_path):
        continue

    # Download from HF cache (or repo)
    local_file = hf_hub_download(
        repo_id=repo_id,
        filename=hf_path.split("/")[-2] + "/" + filename,  # keep subfolder
        repo_type="dataset"
    )

    # Cross-drive safe copy
    shutil.copy(local_file, save_path)


# noice data
out_dir = "data/background_noise"
os.makedirs(out_dir, exist_ok=True)

# 1) Get file list from GitHub API
api = "https://api.github.com/repos/karolpiczak/ESC-50/contents/audio"
files = [f["name"] for f in requests.get(api).json() if f["name"].endswith(".wav")]

base = "https://raw.githubusercontent.com/karolpiczak/ESC-50/master/audio/"

def dl(fname):
    path = os.path.join(out_dir, fname)
    if os.path.exists(path): return
    r = requests.get(base + fname, stream=True)
    if r.status_code == 200:
        with open(path, "wb") as f:
            for c in r.iter_content(8192):
                f.write(c)

# 2) Parallel download (SUPERFAST)
with ThreadPoolExecutor(max_workers=16) as ex:
    list(tqdm(ex.map(dl, files), total=len(files)))


Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

270it [00:00, 780.41it/s]
100%|██████████| 1000/1000 [00:00<00:00, 65235.31it/s]


## Step 3: Configure and Train the Model

Now for the fun part! We will create a `config.yaml` file and then run the Nanowakeword training command.

In [13]:
# @title Step 3.1: Configuración Rápida (Reusando datos)
import yaml

config_dict = {
    "model_name": "Kira",
    "output_dir": "./trained_models",
    "target_phrase": "kira",

    # --- RUTAS RESTAURADAS QUE EL MOTOR EXIGÍA ---
    "positive_data_path": "./data/generated_positive",
    "negative_data_path": "./data/generated_negative",
    "background_paths": ["./data/background_noise"],
    "rir_paths": ["./data/rir"],

    "model_type": "dnn",
    "layer_size": 128,
    "n_blocks": 3,
    "dropout_prob": 0.2,
    "steps": 4000,  # Entrenamiento súper rápido
    "batch_size": 128,
    "optimizer_type": "adamw",
    "learning_rate_max": 0.001,
    "lr_scheduler_type": "onecycle",
    "weight_decay": 0.01,
    "momentum": 0.9,
    "num_workers": 3,

    # TRUCO: Le damos los mismos archivos como validación para que no falle
    "feature_manifest": {
        "targets": {"t": "./trained_models/Kira/features/positive_features_train.npy"},
        "targets_val": {"t": "./trained_models/Kira/features/positive_features_train.npy"},
        "negatives": {"n": "./trained_models/Kira/features/hard_negative_features.npy"},
        "negatives_val": {"n": "./trained_models/Kira/features/hard_negative_features.npy"},
        "backgrounds": {"b": "./trained_models/Kira/features/pure_noise_features.npy"},
        "backgrounds_val": {"b": "./trained_models/Kira/features/pure_noise_features.npy"}
    },

    "checkpoint_averaging_top_k": 1,
    "checkpointing": {"enabled": True, "interval_steps": 1000, "limit": 2},
    "early_stopping_patience": 0,
    "main_delta": 0.0001,
    "WARMUP_STEPS": 500,
    "onnx_opset_version": 17,
    "show_training_summary": True,
    "debug_mode": True,
    "ema_alpha": 0.01,

    # ¡CRÍTICO! Apagamos esto para que re-use los archivos de anoche y no tarde horas
    "generate_clips": False,
    "transform_clips": False,
    "train_model": True,
    "overwrite": True
}

config_path = "./config.yaml"
with open(config_path, "w") as f:
    yaml.dump(config_dict, f, default_flow_style=False, sort_keys=False)
print("✅ config.yaml corregido. ¡A entrenar!")

✅ config.yaml corregido. ¡A entrenar!


**Run Training!**

In [14]:
# @title Step 3.2: Run the Magic Command! 🚀
# This command will do everything: augment data, extract features, and train the model.
# It might take some time depending on the hardware (especially on a CPU).

from nanowakeword.trainer import train

args_list = [
    '--config_path', f'{config_path}',
]

print("Starting NanoWakeWord training...")

try:
    train(args_list)
    print("\n\nCONGRATULATIONS! (✿◕‿◕✿)")
    print("Your custom wake word model has been successfully trained!")

except Exception as e:
    print(f"\nAn error occurred during training: {e}")

Starting NanoWakeWord training...


  _   _               __          __   _     __          __           _ 
 | \ | |              \ \        / /  | |    \ \        / /          | |
 |  \| | __ _ _ __   __\ \  /\  / /_ _| | ____\ \  /\  / /__  _ __ __| |
 | . ` |/ _` | '_ \ / _ \ \/  \/ / _` | |/ / _ \ \/  \/ / _ \| '__/ _` |
 | |\  | (_| | | | | (_) \  /\  / (_| |   <  __/\  /\  / (_) | | | (_| |
 |_| \_|\__,_|_| |_|\___/ \/  \/ \__,_|_|\_\___| \/  \/ \___/|_|  \__,_|

STEP 8: Verifying and Preprocessing Data Directories

====================================================

INFO: Data in 'generated_positive' has changed. Re-verifying...

...

INFO: ONNX model saved successfully.

INFO: Saving final PyTorch model (state_dict) to '/content/trained_models/Kira/model/Kira.pt'

INFO: PyTorch model saved successfully.

INFO: Master training journal updated successfully at '/content/trained_models/training_journal.md'



CONGRATULATIONS! (✿◕‿◕✿)
Your custom wake word model has been successfully trained!


## What's Next?

You have successfully trained your own custom wake word model!

You can now download the `.onnx` file from the `trained_models` directory (check the file browser on the left) and use it in your own applications.

For more advanced topics, such as using your own datasets or fine-tuning the configuration, please check out our full documentation on **[GitHub](https://github.com/arcosoph/nanowakeword)**.

---
## Step 4: Save Your Model to Google Drive

The final step is to save your trained model and performance graph to a safe and accessible place. Instead of a slow direct download, we will save the files directly to your Google Drive. This process is almost instantaneous.

Run the cells below to:
1.  Connect your Google Drive account.
2.  Copy all the trained files into a new folder named `nanowakeword_models` in your Drive.

In [20]:
from google.colab import files
import os

ruta_modelo = "/content/trained_models/Kira/model/Kira.onnx"

if os.path.exists(ruta_modelo):
    print("¡Archivo encontrado! Preparando descarga directa...")
    files.download(ruta_modelo)
else:
    print("❌ Error: No se encuentra el archivo en la memoria. ¿Se reinició la máquina virtual?")

¡Archivo encontrado! Preparando descarga directa...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
# @title Step 4.2: Copy Final Model and Artifacts to Google Drive 📂

import os
import shutil

# --- Configuration ---
# Get model_name and output_dir from the config_dict defined earlier
model_name = config_dict.get("model_name", "my_model")
output_dir = config_dict.get("output_dir", "./trained_models")

# --- Source and Destination Paths ---
# The source project directory containing all generated files
source_project_dir = os.path.join(output_dir, model_name)

# The destination folder in your Google Drive
drive_destination_dir = f"drive/MyDrive/nanowakeword_models/{model_name}"

# --- Start Copy Process ---
print("Starting the process to copy trained files to Google Drive...")

# Check if the source directory exists
if not os.path.exists(source_project_dir):
    print(f"\nERROR: Source directory not found at '{source_project_dir}'")
    print("This indicates that the training process did not create the expected output folder.")
    print("Please ensure the training step completed successfully before running this cell.")
else:
    # If an old folder exists in Drive, remove it to ensure a clean copy
    if os.path.exists(drive_destination_dir):
        print(f"🔄 Found an existing folder in Drive. Removing it for a fresh copy: '{drive_destination_dir}'")
        shutil.rmtree(drive_destination_dir)

    # --- Copy the entire project folder ---
    # This is much simpler and more reliable than copying individual files.
    # It preserves the professional directory structure.
    try:
        shutil.copytree(source_project_dir, drive_destination_dir)

        print("\n" + "="*50)
        print("✅ SUCCESS! All files have been saved to your Google Drive.")
        print("="*50)
        print(f"\nYour complete project, including the model and performance graphs, can be found in:")
        print(f"➡️ '{drive_destination_dir}'")

        # Optional: List the contents of the new folder in Drive for verification
        print("\nContents of the saved folder:")
        for root, dirs, files in os.walk(drive_destination_dir):
            level = root.replace(drive_destination_dir, '').count(os.sep)
            indent = ' ' * 4 * (level)
            print(f"{indent}{os.path.basename(root)}/")
            sub_indent = ' ' * 4 * (level + 1)
            for f in files:
                print(f"{sub_indent}{f}")

    except Exception as e:
        print(f"\nERROR: An unexpected error occurred during the copy process.")
        print(f"Details: {e}")

Starting the process to copy trained files to Google Drive...
🔄 Found an existing folder in Drive. Removing it for a fresh copy: 'drive/MyDrive/nanowakeword_models/Kira'

✅ SUCCESS! All files have been saved to your Google Drive.

Your complete project, including the model and performance graphs, can be found in:
➡️ 'drive/MyDrive/nanowakeword_models/Kira'

Contents of the saved folder:
Kira/
    training_artifacts/
        checkpoints/
            checkpoint_step_19000.pth
            checkpoint_step_18000.pth
        training_debug/
            training_debug.log
        graphs/
            training_performance_graph.png
    features/
        positive_features_train.npy
        pure_noise_features.npy
        hard_negative_features.npy
    model/
        Kira.pt
        Kira.onnx
